# Lektion 11 - Agent-zu-Agent (A2A) Protokoll


## Einrichtung


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv

In [ ]:
import os
import dotenv
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Was ist das A2A-Protokoll?

Das **Agent-to-Agent (A2A) Protokoll** ist ein offener Standard, der es KI-Agenten ermöglicht, zu kommunizieren,
sich zu entdecken und zusammenzuarbeiten – selbst wenn sie auf unterschiedlichen Frameworks basieren oder von
verschiedenen Diensten gehostet werden.

Wichtige Konzepte:

- **Entdeckung** – Agenten veröffentlichen eine *Agent Card*, die ihre Fähigkeiten beschreibt, wodurch es
  anderen Agenten (oder Orchestratoren) leicht gemacht wird, den richtigen Spezialisten für eine Aufgabe zu finden.
- **Nachrichtenübermittlung** – Agenten tauschen strukturierte Nachrichten über ein gemeinsames Protokoll aus, sodass eine
  Anfrage von einem Agenten von einem anderen verstanden und erfüllt werden kann, unabhängig von dessen interner
  Implementierung.
- **Aufgaben-Lebenszyklus** – A2A definiert Zustände wie *übermittelt*, *in Arbeit*, *abgeschlossen* und
  *fehlgeschlagen*, was dem Orchestrator volle Einsicht darüber gibt, wie eine delegierte Aufgabe voranschreitet.

In dieser Lektion simulieren wir A2A-ähnliche Zusammenarbeit, indem wir drei spezialisierte Reiseagenten
in einen Workflow einbinden, bei dem jeder Agent seine Expertise einbringt und Ergebnisse an den nächsten weitergibt.


## Erstellung spezialisierter Reiseveranstalter


In [ ]:
currency_agent = client.as_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = client.as_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = client.as_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Multi-Agenten-Zusammenarbeit über Workflow

Wir verbinden die drei Agenten in einem sequentiellen Workflow, der den A2A-Nachrichtenaustausch widerspiegelt:

1. **CurrencyExchangeAgent** erhält die Benutzeranfrage und gibt Währungsempfehlungen.
2. **ActivityPlannerAgent** erhält den erweiterten Kontext und fügt Aktivitätsempfehlungen hinzu.
3. **TravelManagerAgent** fasst beide Eingaben zu einer endgültigen Reiseübersicht zusammen.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Verständnis von A2A in der Produktion

In einer Produktionsumgebung eröffnet das A2A-Protokoll leistungsstarke Szenarien über Service-Grenzen hinweg:

| Fähigkeit | Beschreibung |
|---|---|
| **Framework-übergreifende Interoperabilität** | Ein Agent, der mit einem Framework erstellt wurde, kann Aufgaben an einen Agenten delegieren, der mit einem anderen A2A-konformen Framework erstellt wurde, was eine echte organisationsübergreifende Interoperabilität ermöglicht. |
| **Service-Grenzen** | Agenten können in separaten Microservices, Cloud-Regionen oder sogar verschiedenen Organisationen leben und dennoch nahtlos zusammenarbeiten. |
| **Dynamische Entdeckung** | Ein Orchestrator kann zur Laufzeit ein Agent Card-Register abfragen, um den am besten geeigneten Spezialisten für eine Teilaufgabe zu finden. |
| **Streaming & Push-Benachrichtigungen** | A2A unterstützt Server-Sent Events (SSE) für Echtzeit-Fortschrittsupdates und Push-Benachrichtigungen für lang laufende Aufgaben. |

Der oben erstellte Workflow ist eine vereinfachte, im Prozess ablaufende Version dieses Musters. In einer echten
Bereitstellung würde jeder Agent einen HTTP-Endpunkt bereitstellen, eine Agent Card veröffentlichen und
über das A2A JSON-RPC-Protokoll kommunizieren.


## Zusammenfassung

In dieser Lektion haben Sie gelernt:

1. **Was das A2A-Protokoll ist** — ein offener Standard für Agent-zu-Agent-Erkennung, Messaging
   und Aufgabenverwaltung.
2. **Wie man spezialisierte Agenten erstellt** — ein Währungsumtausch-Agent, ein Aktivitätsplaner-Agent
   und einen Reise-Manager-Orchestrator.
3. **Wie man Agenten in einen Workflow einbindet** — mit `WorkflowBuilder`, um sequenzielle
   Nachrichtenweitergabe zwischen Agenten zu modellieren.
4. **Wie A2A in der Praxis funktioniert** — Ermöglichung von plattform- und dienstübergreifender Zusammenarbeit
   mit dynamischer Erkennung und Streaming-Updates.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Haftungsausschluss**:
Dieses Dokument wurde mit dem KI-Übersetzungsdienst [Co-op Translator](https://github.com/Azure/co-op-translator) übersetzt. Obwohl wir uns um Genauigkeit bemühen, beachten Sie bitte, dass automatisierte Übersetzungen Fehler oder Ungenauigkeiten enthalten können. Das Originaldokument in seiner Ursprungssprache gilt als maßgebliche Quelle. Bei kritischen Informationen wird eine professionelle menschliche Übersetzung empfohlen. Wir übernehmen keine Haftung für Missverständnisse oder Fehlinterpretationen, die aus der Verwendung dieser Übersetzung entstehen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
